# Step 2 — Build the Multi3Hate table and extract multilingual features

**Kaggle settings (right panel):** Accelerator = **GPU T4**, Internet = **On**.

This notebook:
1. downloads Multi3Hate from the authors' GitHub,
2. builds one row per meme–culture pair (1,500 rows) with the **native-language meme image**, the **native-language caption**, the official label and the **annotator vote counts** (for soft labels),
3. extracts features with **SigLIP 2** (multilingual) and **OpenCLIP ViT-B/32** (your original backbone) for comparison,
4. checks whether culture is recoverable from each feature set.

Outputs in `/kaggle/working/`: `multi3hate.csv`, `feats_siglip2.npz`, `feats_openclip_b32.npz`.
Use **Save Version → Save & Run All** so Step 3 can load these files as input.

In [ ]:
!pip install -q -U "transformers>=4.49" open_clip_torch

In [ ]:
import os, subprocess
REPO = "/kaggle/working/Multi3Hate"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/MinhDucBui/Multi3Hate.git", REPO], check=True)
print(os.listdir(f"{REPO}/data"))

In [ ]:
import glob
import numpy as np
import pandas as pd

DATA = f"{REPO}/data"
LANG2CULT = {"en": "US", "de": "DE", "es": "MX", "hi": "IN", "zh": "CN"}

final = pd.read_csv(f"{DATA}/final_annotations.csv")        # official aggregated labels
raw = pd.read_csv(f"{DATA}/raw_annotations.csv")            # one row per annotator judgement
raw = raw.dropna(subset=["hatespeech"])
votes = (raw.groupby(["dataset_language", "Meme ID"])["hatespeech"]
            .agg(n_hate="sum", n_total="count").reset_index())

# image path per (language, meme id): data/memes/<lang>/<template>/<id>.jpg
img_index = {}
for p in glob.glob(f"{DATA}/memes/*/*/*"):
    lang = p.split("/")[-3]
    mid = int(os.path.splitext(os.path.basename(p))[0])
    img_index[(lang, mid)] = p

rows = []
for lang, cult in LANG2CULT.items():
    cap = pd.read_csv(f"{DATA}/captions/{lang}.csv")
    for _, r in cap.iterrows():
        mid = int(r["Meme ID"])
        v = votes[(votes.dataset_language == lang) & (votes["Meme ID"] == mid)]
        rows.append({
            "meme_id": mid,
            "culture": cult,
            "language": lang,
            "image_path": img_index[(lang, mid)],
            "text": str(r["Translation"]).replace("<sep>", " / "),
            "n_hate": int(v["n_hate"].iloc[0]),
            "n_total": int(v["n_total"].iloc[0]),
            "label": int(final.loc[final["Meme ID"] == mid, cult].iloc[0]),
        })
df = pd.DataFrame(rows).sort_values(["meme_id", "culture"]).reset_index(drop=True)
df.to_csv("/kaggle/working/multi3hate.csv", index=False)

print(df.shape, "| memes:", df.meme_id.nunique(), "| annotations:", df.n_total.sum())
print("\nOfficial hate prevalence per culture (compare with Table 1 of your paper):")
print(df.groupby("culture")["label"].mean().round(3).to_string())
agree = (df.label == (df.n_hate / df.n_total > 0.5)).mean()
print(f"\nOfficial label == simple vote majority on {agree:.1%} of rows (rest are ties/other rule)")
df.head()

In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

def as_tensor(out):
    """transformers v5 returns a model-output object from get_*_features; older versions return a tensor."""
    if isinstance(out, torch.Tensor):
        return out
    for key in ("pooler_output", "image_embeds", "text_embeds"):
        v = getattr(out, key, None)
        if isinstance(v, torch.Tensor):
            return v
    raise TypeError(f"Unexpected feature output type: {type(out)}")

def load_backend(name):
    if name == "siglip2":
        from transformers import AutoModel, AutoProcessor
        mid = "google/siglip2-base-patch16-224"
        model = AutoModel.from_pretrained(mid).to(DEVICE).eval()
        proc = AutoProcessor.from_pretrained(mid)
        def enc_img(imgs):
            return as_tensor(model.get_image_features(**proc(images=imgs, return_tensors="pt").to(DEVICE)))
        def enc_txt(texts):
            # SigLIP 2 was trained with lowercased text, max_length=64 padding
            inp = proc(text=[t.lower() for t in texts], padding="max_length", max_length=64,
                       truncation=True, return_tensors="pt").to(DEVICE)
            return as_tensor(model.get_text_features(**inp))
        return enc_img, enc_txt
    if name == "openclip":
        import open_clip
        model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
        model = model.to(DEVICE).eval()
        tok = open_clip.get_tokenizer("ViT-B-32")
        def enc_img(imgs):
            return model.encode_image(torch.stack([preprocess(i) for i in imgs]).to(DEVICE))
        def enc_txt(texts):
            return model.encode_text(tok(texts).to(DEVICE))
        return enc_img, enc_txt
    raise ValueError(name)

@torch.no_grad()
def extract(df, backend, out_path, batch=32):
    if os.path.exists(out_path):
        print(f"[skip] {out_path} already exists")
        return
    enc_img, enc_txt = load_backend(backend)
    img, txt = [], []
    for i in range(0, len(df), batch):
        chunk = df.iloc[i:i + batch]
        ims = [Image.open(p).convert("RGB") for p in chunk.image_path]
        img.append(F.normalize(enc_img(ims).float(), dim=-1).cpu().numpy())
        txt.append(F.normalize(enc_txt(chunk.text.tolist()).float(), dim=-1).cpu().numpy())
    img, txt = np.concatenate(img), np.concatenate(txt)
    np.savez(out_path, X=np.concatenate([img, txt], 1).astype(np.float32), img_dim=img.shape[1],
             culture=df.culture.values.astype(str), meme_id=df.meme_id.values.astype(str),
             y_soft=(df.n_hate / df.n_total).values.astype(np.float32),
             y_hard=df.label.values.astype(np.int64), backend=backend)
    print(f"saved {out_path}: {img.shape[0]} x {img.shape[1] + txt.shape[1]}")
    del enc_img, enc_txt
    torch.cuda.empty_cache()

In [ ]:
extract(df, "siglip2",  "/kaggle/working/feats_siglip2.npz")
extract(df, "openclip", "/kaggle/working/feats_openclip_b32.npz")

## Check: is culture recoverable from the new features?

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score


def diagnose_features(path):
    z = np.load(path, allow_pickle=True)
    X = z["X"].astype(np.float32)
    cult = z["culture"].astype(str)
    meme = z["meme_id"].astype(str)
    img_dim = int(z["img_dim"]) if "img_dim" in z else None

    print("\n" + "=" * 70)
    print("C. ARE INPUTS DIFFERENT ACROSS CULTURES FOR THE SAME MEME?")
    print("=" * 70)
    parts = {"full": X}
    if img_dim:
        parts["image"], parts["text"] = X[:, :img_dim], X[:, img_dim:]
    for name, A in parts.items():
        An = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
        dists = []
        for mid in np.unique(meme):
            rows = An[meme == mid]
            if len(rows) > 1:
                sim = rows @ rows.T
                iu = np.triu_indices(len(rows), 1)
                dists.append(np.mean(1 - sim[iu]))
        med = float(np.median(dists))
        flag = "IDENTICAL across cultures" if med < 1e-4 else "differs across cultures"
        print(f"  {name:6s}: median within-meme cosine distance = {med:.5f} -> {flag}")

    print("\n" + "=" * 70)
    print("D. CAN CULTURE BE PREDICTED FROM THE RAW INPUT? (grouped by meme)")
    print("=" * 70)
    for name, A in parts.items():
        acc = cross_val_score(LogisticRegression(max_iter=3000), A, cult,
                              groups=meme, cv=GroupKFold(5)).mean()
        print(f"  {name:6s}: culture probe accuracy = {acc:.3f} (chance = 0.200)")
    print("\n  Interpretation:")
    print("   ~0.20 -> inputs carry no culture signal: the adversary has nothing to")
    print("            remove, and a culture-invariant model must give identical")
    print("            outputs for all 5 cultures of a meme.")
    print("   ~1.00 -> culture (language) is trivially readable from the input; a")
    print("            working adversary/probe should start far above chance.")

In [ ]:
for f in ["/kaggle/working/feats_openclip_b32.npz", "/kaggle/working/feats_siglip2.npz"]:
    print("\n\n############", os.path.basename(f), "############")
    diagnose_features(f)

In [ ]:
# keep the output small: remove the cloned repo (images are not needed after extraction)
import shutil; shutil.rmtree(REPO, ignore_errors=True)
print(os.listdir("/kaggle/working"))

## What to send back
The printed per-culture prevalence table and the two diagnosis blocks (sections C–D). Then open **Step 3**.